# Swin Transformer v2 画像・タグ学習システム (Google Colab版)

Google Drive 内の **Cryptomator 暗号化保管庫 (Vault)** から **WebDAV** を介してインメモリでストリーミング学習を行います。

> **💡 コスト節約のための推奨手順**:
> Google Colab の貴重な GPU 時間を無駄に消費しないため、**まずは通常の「CPU ランタイム」で実行**し、Google Drive のマウントと Cryptomator の復号・WebDAV 疎通が正常に完了することを確認してください。
> 接続確認が取れた後、メニューの「ランタイムのタイプを変更」で **GPU** に切り替えて本格的な学習・推論に進むことができます。

--- 
## 【フェーズ 1: 事前検証 (CPU ランタイムで実行可能)】
Google Drive と Cryptomator の設定が正常に機能するか確認します。

### 1. Google Drive のマウント & Vault フォルダ事前診断

In [4]:
from google.colab import drive
import os

# 1. Google Drive をマウント
drive.mount('/content/drive')

# 2. Google Drive 内の Vault (保管庫) フォルダパスを指定してください
# 例: '/content/drive/MyDrive/MyVault' など
VAULT_PATH = '/content/drive/MyDrive/hentai/hentai'  # ← あなたの保管庫パスに変更してください

# 事前診断チェック
print(f"\n--- 保管庫パスの診断: {VAULT_PATH} ---")
if os.path.exists(VAULT_PATH):
    print("✅ 保管庫フォルダが存在することを確認しました。")
    files = os.listdir(VAULT_PATH)
    has_vault_config = any("vault.cryptomator" in f or "masterkey.cryptomator" in f for f in files)
    if has_vault_config:
        print("✅ Cryptomator 保管庫の設定ファイル (vault.cryptomator) を検出しました。")
    else:
        print("⚠️ 警告: フォルダ内に Cryptomator の設定ファイルが見当たりません。パスが保管庫のルートフォルダを指しているか確認してください。")
else:
    print("❌ エラー: 指定された Vault パスが見つかりません。Google Drive のフォルダ名や大文字小文字を確認してください。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- 保管庫パスの診断: /content/drive/MyDrive/hentai/hentai ---
✅ 保管庫フォルダが存在することを確認しました。
✅ Cryptomator 保管庫の設定ファイル (vault.cryptomator) を検出しました。


### 2. 環境構築 & Cryptomator CLI のセットアップ

In [8]:
# 1. 必要な Python パッケージのインストール
!pip install -q timm rule34Py python-dotenv pyyaml requests pillow tqdm

# 2. Cryptomator CLI のインストール (v0.6.2: JRE同梱スタンドアロン版)
!mkdir -p /opt
!wget -q -O /tmp/cryptomator-cli.zip https://github.com/cryptomator/cli/releases/download/0.6.2/cryptomator-cli-0.6.2-linux-x64.zip
!unzip -q -o /tmp/cryptomator-cli.zip -d /opt
!chmod +x /opt/cryptomator-cli/bin/cryptomator-cli
!ln -sf /opt/cryptomator-cli/bin/cryptomator-cli /usr/local/bin/cryptomator-cli
!rm -f /tmp/cryptomator-cli.zip

# 3. インストール確認
!cryptomator-cli --version


0.6.2


### 3. Vault アンロック & WebDAV 疎通テスト（ヘルスチェック）

パスワードを入力して WebDAV を起動し、正しくデータセットが読み取れるか検証します。

In [ ]:
import getpass
import os
import re
import subprocess
import time
import requests
import yaml

if not os.path.exists(VAULT_PATH):
    raise FileNotFoundError(f"Vault パスが存在しません: {VAULT_PATH}")

# パスワード入力 (画面に平文表示されません)
vault_password = getpass.getpass("Cryptomator Vault のパスワードを入力してください: ")

# 以前のプロセスが実行中の場合は終了
if 'crypto_proc' in globals() and crypto_proc.poll() is None:
    crypto_proc.terminate()
    time.sleep(1)

log_path = "/tmp/cryptomator.log"
if os.path.exists(log_path):
    os.remove(log_path)

# パスワードを一時ファイルに安全に保存（環境変数経由の引数解析バグを完全回避）
pw_file_path = "/tmp/vault.pass"
with open(pw_file_path, "w", encoding="utf-8") as f:
    f.write(vault_password)
os.chmod(pw_file_path, 0o600)

print("Cryptomator WebDAV サーバーをバックグラウンド起動中...")
# -v: 詳細ログ出力（デフォルトでは org.cryptomator のログが無効化されているため必須）
# --loopbackPort 42427: ポート番号を 42427 に固定
cmd = [
    "cryptomator-cli",
    "-v",
    "unlock",
    "--mounter", "org.cryptomator.frontend.webdav.mount.FallbackMounter",
    "--loopbackPort", "42427",
    "--password:file", pw_file_path,
    VAULT_PATH
]

# ログファイルに出力をリダイレクト
log_file = open(log_path, "w", encoding="utf-8")
crypto_proc = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT
)

# 起動直後にパスワード一時ファイルを安全に削除
time.sleep(1.5)
if os.path.exists(pw_file_path):
    try:
        os.remove(pw_file_path)
    except Exception:
        pass

webdav_url = None
start_time = time.time()
printed_lines = 0

print("起動ログをリアルタイム監視中...")
while time.time() - start_time < 45:
    if os.path.exists(log_path):
        with open(log_path, "r", encoding="utf-8", errors="ignore") as f:
            lines = f.readlines()
            for line in lines[printed_lines:]:
                print("  [Cryptomator]", line.strip())
                match = re.search(r'(http://(?:localhost|127\.0\.1):\d+/[^\s]+)', line)
                if match:
                    webdav_url = match.group(1).rstrip("/")
            printed_lines = len(lines)

    if webdav_url:
        break

    # プロセスが異常終了していないか確認
    if crypto_proc.poll() is not None:
        print(f"\n❌ Cryptomator プロセスが終了しました (終了コード: {crypto_proc.returncode})")
        if os.path.exists(log_path):
            with open(log_path, "r", encoding="utf-8", errors="ignore") as f:
                print("--- ログ出力 ---")
                print(f.read())
        raise RuntimeError("Cryptomator の起動に失敗しました。上記ログを確認してください。")

    time.sleep(0.5)

# 万が一ログから URL が検出できなかった場合のフォールバック（固定ポートから探査）
if not webdav_url:
    print("ログから URL を直接検出できなかったため、固定ポート (42427) を探査中...")
    vault_name = os.path.basename(os.path.normpath(VAULT_PATH))
    candidate_urls = [
        f"http://localhost:42427/{vault_name}",
        f"http://127.0.0.1:42427/{vault_name}",
        "http://localhost:42427",
        "http://127.0.0.1:42427"
    ]
    for cand in candidate_urls:
        try:
            r = requests.request("PROPFIND", cand, headers={"Depth": "0"}, timeout=2)
            if r.status_code in (200, 207, 401, 403):
                webdav_url = cand
                print(f"✅ 固定ポート探査により WebDAV サービスを検知しました: {webdav_url}")
                break
        except Exception:
            pass

if not webdav_url:
    if os.path.exists(log_path):
        with open(log_path, "r", encoding="utf-8", errors="ignore") as f:
            print("--- タイムアウト時のログ出力 ---")
            print(f.read())
    raise TimeoutError("WebDAV URL を検出できませんでした。上記ログを確認してください。")

print(f"\n✅ [検知成功] WebDAV URL: {webdav_url}")

# WebDAV 疎通ヘルスチェック (PROPFIND)
try:
    resp = requests.request("PROPFIND", webdav_url, headers={"Depth": "1"}, timeout=10)
    if resp.status_code in (200, 207):
        print("✅ WebDAV サーバーへの HTTP 通信テストに成功しました！")
    else:
        print(f"⚠️ 警告: WebDAV 応答ステータス: {resp.status_code}")
except Exception as e:
    print(f"❌ WebDAV 通信エラー: {e}")

# config.yaml を自動更新
config_path = "config.yaml"
if not os.path.exists(config_path) and os.path.exists("example.config.yaml"):
    with open("example.config.yaml", "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
else:
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}

cfg.setdefault("storage", {})["webdav_url"] = webdav_url
with open(config_path, "w", encoding="utf-8") as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)
print("✅ config.yaml に接続 URL を反映しました。\n")


### 4. 保管庫内データセットの読み取り確認

In [ ]:
# 保存済みデータセット一覧を表示
!python main.py list

print("\n" + "=" * 60)
print("🎉 [確認完了] Google Drive と Cryptomator の連携テストがすべて成功しました！")
print("=" * 60)
print("\n【次のステップ】")
print("1. 上部メニューの [ランタイム] -> [ランタイムのタイプを変更] を選択します。")
print("2. ハードウェア アクセラレータで [GPU (T4 など)] を選択して保存します。")
print("3. 下記の【フェーズ 2】へ進み、GPU での高速学習・推論を開始してください！")

--- 
## 【フェーズ 2: GPU での学習 & 推論】
GPU ランタイムに切り替えた後、以下のセルを実行します。

### 5. GPU の認識確認

In [ ]:
!nvidia-smi

### 6. Swin Transformer v2 学習の実行 (GPU 高速化・SSD非保存)

Cryptomator の WebDAV から画像を直接 RAM 上に並列プリロードし、GPU で学習を実行します。
Colab のローカルディスクには平文画像は一切書き込まれません。

In [ ]:
# 省メモリモード (low_vram) または 高精度モード (high_quality) を指定可能
!python main.py train \
    --dataset-name "test_overwatch" \
    --preset high_quality \
    --min-frequency 5 \
    --epochs 20 \
    --batch-size 32

### 7. 学習済みモデルによる推論（タグ予測）

学習したチェックポイント（`.pth`）を使って、画像ファイルまたは Web 上の画像 URL からタグを予測します。

In [ ]:
# テスト画像 URL または Google Drive 内の画像パスを指定
TEST_IMAGE_URL = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"

!python main.py predict \
    --checkpoint "./checkpoints/test_overwatch_best_model.pth" \
    --image "{TEST_IMAGE_URL}" \
    --threshold 0.35 \
    --top-k 15

### 8. クリーンアップ (Cryptomator の安全なロック・終了)

作業終了後、バックグラウンドの WebDAV サーバーを停止し、Vault を安全にロックします。

In [ ]:
try:
    if 'crypto_proc' in globals() and crypto_proc.poll() is None:
        crypto_proc.terminate()
        crypto_proc.wait(timeout=5)
        print("Cryptomator WebDAV サーバーを正常に停止し、Vault をロックしました。")
    else:
        print("Cryptomator プロセスは実行されていません。")
except Exception as e:
    print(f"停止時エラー: {e}")